# LUCiD quickstart — simulate one event and see it

LUCiD is a **JAX, fully-differentiable** simulation of optical-photon propagation in
particle detectors. This 5-minute tour does the simplest thing: shoot one muon through a
SK-like water Cherenkov tank, simulate the light, and display the per-PMT charge.

Everything is one library call away:

| step | seam |
|---|---|
| build the detector + forward model | `simulation.setup_event_simulator` |
| define a particle track | `detector_params.ParticleParams` |
| draw the event | `visualization.create_detector_display` |

## 1. Build the simulator

`setup_event_simulator` reads a **geometry** JSON (shape + PMT layout) and a **physics**
JSON (optical properties), and returns a JIT-compiled function `(track, key) -> hits`.
We use the full sensor grid so every PMT is reachable (a coarse grid would leave white
holes in the display).

In [ ]:
import sys; sys.path.append('..')
import jax, numpy as np
from lucid.simulation import setup_event_simulator
from lucid.detector_params import ParticleParams
from lucid.visualization import create_detector_display

GEOM, PHYS = '../config/SK_like_geom_config.json', '../config/SK_like_physics_config.json'
sim = setup_event_simulator(GEOM, 200_000, K=4, physics_config=PHYS, default_detector_params=True,
                            particle='muon', n_cap=120, n_angular=180, n_height=120)
print('simulator ready')

## 2. Shoot a muon and simulate the light

A track is just an energy, a vertex, a direction and a start time. The simulator returns
the per-PMT charge (photo-electrons) — the Cherenkov ring.

In [ ]:
track = ParticleParams.from_cartesian(energy=1000., position=[0., 0., 0.],
                                      direction=[1., 0., 0.], t0=0.)
charge = np.asarray(sim(track, jax.random.PRNGKey(0))[3])      # (n_sensors,) per-PMT charge (pe)
print(f'{(charge > 0).sum()} of {charge.size} PMTs lit | total charge {charge.sum():.0f} pe')

## 3. Display the event

`create_detector_display` unrolls the cylinder (barrel + end caps) into a 2D map. The
Cherenkov ring of the muon shows up immediately.

In [ ]:
display = create_detector_display(GEOM, sparse=False)
display(charge, np.zeros_like(charge), file_name=None, perc_min=0.0, perc_max=99.5)

## Where to go next

- **Reconstruct** the track back from this ring → `reconstruction` notebooks (`fitting.fit_track_multistart`).
- **Calibrate** the detector's optics from controlled light → `calibration_optimization`.
- **See the gradients** that make all of this work → the gradient notebooks.

The forward model here is differentiable end-to-end — every later notebook is just a
gradient of this same `setup_event_simulator` call.